# Dealer Gamma 05 — pinning: does settlement gravitate to the dealer-long-gamma strike?

Pre-registration: `discussions/Dealer_Gamma/001_claude_branch_preregistration.md`. Code: `src/data_engineering/modeling/dealer_gamma.py`.

For every 08:00 UTC expiry: strike table at 05:00 for that expiry (strikes within ±5% of S). Pin strike K by rule (`long_gamma` primary, `long_gamma_noblock`, `max_position`, `short_gamma` control); placebo $K' = 2S_{05} - K$. Settlement proxy X = 07:30–08:00 index average. $\Delta = (|X-K| - |X-K'|)/S_{05}$ in bp; negative = settled closer to the pin than to the placebo.

**H:** mean Δ < 0 for `long_gamma` and `max_position`; `short_gamma` Δ ≥ 0. Bootstrap over expiries. PASS = both currencies, primary group `all`, rule `long_gamma`.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import dealer_gamma as dg

ENV = dg.notebook_env('05', PROJECT_ROOT)
OUT, CACHE, HIST = ENV['out'], ENV['cache'], ENV['history']
PERIODS = ['discovery'] + (['holdout'] if ENV['holdout'] else [])
print(dg.VERSION, '| synthetic' if ENV['synthetic'] else '', '| periods', PERIODS)
print('data', ENV['data_root']); print('cache', CACHE); print('out', OUT)

In [ ]:
EV, TS = [], []
for c in dg.CURRENCIES:
    d = dg.load_cache(CACHE, c)
    ev = dg.pin_events(d['pins'], d['settle'])
    EV.append(ev)
    for per in PERIODS:
        TS.append(dg.pin_test(ev, per))
EV = pd.concat(EV); TS = pd.concat(TS)
EV.to_csv(OUT / '05_events.csv', index=False, float_format='%.6g'); TS.to_csv(OUT / '05_tests.csv', index=False)
print(EV.groupby(['currency', 'period', 'type']).expiry_date.nunique().unstack(fill_value=0))
display(TS.pivot_table(index=['period', 'group', 'rule'], columns='currency', values=['mean_delta_bp', 'p_one_sided']).round(3))

In [ ]:
prim = TS[(TS.group == 'all') & (TS.rule == 'long_gamma')]
dec = dg.decide(prim, ['group', 'rule'])
secondary = dg.decide(TS[TS.rule.isin(['long_gamma', 'max_position', 'long_gamma_noblock'])], ['group', 'rule'])
dg.dump({'registration': dg.registration(), 'decision_primary': dec, 'secondary': secondary}, OUT / '05_decision.json')
print(json.dumps(dec, indent=2))

In [ ]:
fig, ax = plt.subplots(1, len(dg.CURRENCIES), figsize=(14, 4.5), sharey=True)
order = ['all', 'daily', 'weekly', 'monthly', 'quarterly', 'friday']
for j, c in enumerate(dg.CURRENCIES):
    z = TS[(TS.currency == c) & (TS.period == 'discovery')]
    for k, rule in enumerate(['long_gamma', 'max_position', 'short_gamma']):
        w = z[z.rule == rule].set_index('group').reindex(order)
        ax[j].bar(np.arange(len(order)) + 0.27 * k, w.mean_delta_bp, width=0.27, label=rule)
    ax[j].axhline(0, color='k', lw=0.6); ax[j].set_xticks(np.arange(len(order)) + 0.27, order)
    ax[j].set_title(f'{c} discovery: mean Δ (bp, <0 = pinned)'); ax[j].legend()
plt.tight_layout(); plt.savefig(OUT / '05_pinning.png', dpi=110); plt.show()